# 🧬 KEGG Pathway Analysis for pLLPS Proteins

This notebook analyzes pLLPS (Liquid-Liquid Phase Separation) scores in the context of KEGG pathways.

## Analysis Goals
1. **Score Distribution in Pathways**: Do similar pLLPS scores interact in the same pathway?
2. **Pathway Enrichment**: Are particular pathways enriched or depleted in high/low pLLPS proteins?
3. **Membrane Protein Analysis**: Identify high/low pLLPS membrane proteins and their pathways
4. **Pathway Visualization**: Draw KEGG diagrams annotated with pLLPS scores

## Table of Contents
1. [Setup & Data Loading](#1-setup--data-loading)
2. [Identify High/Low pLLPS Membrane Proteins](#2-identify-highlow-pllps-membrane-proteins)
3. [Fetch KEGG Pathway Data](#3-fetch-kegg-pathway-data)
4. [Pathway Enrichment Analysis](#4-pathway-enrichment-analysis)
5. [Score Similarity Analysis](#5-score-similarity-analysis)
6. [KEGG Pathway Visualization](#6-kegg-pathway-visualization)
7. [Statistical Analysis](#7-statistical-analysis)

**Network Requirements:** This section requires internet access to the KEGG REST API at `rest.kegg.jp`. If running in a restricted environment, you may need to run this notebook in a different environment with network access, or use pre-cached pathway data.

---
## 1. Setup & Data Loading

Import libraries and load the LLPS protein data.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
from pathlib import Path
import re
import requests
import time
from collections import defaultdict, Counter
from typing import List, Dict, Tuple
from bioservices import KEGG
from scipy import stats

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# KEGG API rate limiting: 3 requests per second
KEGG_API_DELAY = 0.34  # seconds between requests

print("✅ Libraries loaded successfully!")
print(f"⏱️  KEGG API rate limit: {1/KEGG_API_DELAY:.1f} requests/second")

In [ ]:
# Initialize KEGG service
kegg = KEGG()
print("✅ KEGG service initialized")

In [ ]:
# Load the LLPS protein data
data_path = Path("Human Phase separation data.xlsx")

if data_path.exists():
    df_raw = pd.read_excel(data_path, engine='openpyxl')
    print(f"✅ Data loaded successfully!")
    print(f"   Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
    print(f"\nColumns: {df_raw.columns.tolist()}")
else:
    print("❌ Data file not found. Please ensure 'Human Phase separation data.xlsx' exists.")
    df_raw = None

In [ ]:
# Display basic statistics
if df_raw is not None:
    print("\n📊 p(LLPS) Score Distribution:")
    print(df_raw['p(LLPS)'].describe())
    
    print("\n🔍 First few rows:")
    display(df_raw.head())

---
## 2. Identify High/Low pLLPS Membrane Proteins

Reuse the function parser from app.py to identify membrane proteins and classify them by pLLPS score.

In [ ]:
# Function categories from app.py - focusing on transmembrane/membrane proteins
MEMBRANE_PATTERNS = [
    r'transmembrane',
    r'integral\s+membrane',
    r'multi[- ]?pass\s+membrane',
    r'single[- ]?pass\s+membrane',
    r'membrane\s+protein',
]

def is_membrane_protein(function_str, protein_name_str=None, location_str=None):
    """
    Determine if a protein is a membrane protein based on function, name, and location.
    
    Parameters
    ----------
    function_str : str
        Function annotation from UniProt
    protein_name_str : str, optional
        Protein name
    location_str : str, optional
        Subcellular location annotation
    
    Returns
    -------
    bool
        True if protein is classified as membrane protein
    """
    # Combine all available text
    combined_text = ''
    if pd.notna(function_str):
        combined_text += str(function_str).lower() + ' '
    if pd.notna(protein_name_str):
        combined_text += str(protein_name_str).lower() + ' '
    if pd.notna(location_str):
        combined_text += str(location_str).lower() + ' '
    
    # Check for membrane patterns
    for pattern in MEMBRANE_PATTERNS:
        if re.search(pattern, combined_text, re.IGNORECASE):
            return True
    
    return False

print("✅ Membrane protein classifier defined")

In [ ]:
# Classify proteins as membrane proteins
if df_raw is not None:
    df = df_raw.copy()
    
    # Add membrane protein classification
    df['is_membrane'] = df.apply(
        lambda row: is_membrane_protein(
            row.get('Function [CC]'),
            row.get('Protein names'),
            row.get('Subcellular location [CC]')
        ),
        axis=1
    )
    
    # Define pLLPS thresholds
    HIGH_PLLPS_THRESHOLD = 0.7
    LOW_PLLPS_THRESHOLD = 0.3
    
    # Classify by pLLPS score
    df['pllps_class'] = pd.cut(
        df['p(LLPS)'],
        bins=[-np.inf, LOW_PLLPS_THRESHOLD, HIGH_PLLPS_THRESHOLD, np.inf],
        labels=['low', 'medium', 'high']
    )
    
    print("\n📊 Classification Results:")
    print(f"Total proteins: {len(df)}")
    print(f"Membrane proteins: {df['is_membrane'].sum()} ({100*df['is_membrane'].sum()/len(df):.1f}%)")
    print(f"\npLLPS Classification:")
    print(df['pllps_class'].value_counts().sort_index())
    
    # High/Low pLLPS membrane proteins
    high_mem = df[(df['is_membrane']) & (df['pllps_class'] == 'high')]
    low_mem = df[(df['is_membrane']) & (df['pllps_class'] == 'low')]
    
    print(f"\n🔴 High pLLPS membrane proteins: {len(high_mem)}")
    print(f"🔵 Low pLLPS membrane proteins: {len(low_mem)}")
    
    # Show examples
    print("\n🔍 High pLLPS membrane protein examples:")
    display(high_mem[['Entry', 'Protein names', 'p(LLPS)']].head(10))
    
    print("\n🔍 Low pLLPS membrane protein examples:")
    display(low_mem[['Entry', 'Protein names', 'p(LLPS)']].head(10))

---
## 3. Fetch KEGG Pathway Data

Retrieve KEGG pathway information for the proteins in our dataset.

**Note:** KEGG API is rate limited to 3 requests per second. This section may take several minutes for large datasets.

**Network Requirements:** This section requires internet access to the KEGG REST API at `rest.kegg.jp`. If running in a restricted environment, you may need to run this notebook in a different environment with network access, or use pre-cached pathway data.

### Troubleshooting UniProt to KEGG Mapping

The conversion from UniProt IDs to KEGG IDs may not succeed for all proteins because:

1. **Not all proteins are in KEGG**: KEGG focuses on metabolic and signaling pathways. Structural proteins, some regulatory proteins, and proteins without well-characterized pathway roles may not be included.

2. **ID conversion methods**: This notebook tries multiple conversion methods:
   - Direct conversion with `uniprot:` prefix
   - Alternative conversion with `up:` prefix  
   - KEGG find service as fallback

3. **Expected mapping rate**: Typically 30-60% of proteins will successfully map to KEGG, depending on the protein set.

4. **Diagnostic output**: The notebook provides mapping statistics to show how many proteins were successfully found in KEGG vs. not found.


In [ ]:
def get_kegg_pathways_for_protein(uniprot_id, kegg_service, cache=None, delay=KEGG_API_DELAY):
    """
    Get KEGG pathways for a UniProt ID.
    
    This function tries multiple methods to convert UniProt IDs to KEGG IDs:
    1. Direct conversion using 'uniprot:' prefix
    2. Alternative conversion using 'up:' prefix
    3. Find service as fallback
    
    Parameters
    ----------
    uniprot_id : str
        UniProt accession ID
    kegg_service : KEGG
        Initialized KEGG service object
    cache : dict
        Cache to avoid redundant API calls
    delay : float
        Delay between API calls in seconds (respects rate limit)
    
    Returns
    -------
    list
        List of KEGG pathway IDs
    """
    if cache is None:
        cache = {}
    
    if uniprot_id in cache:
        return cache[uniprot_id]
    
    kegg_ids = []
    
    try:
        # Method 1: Try standard uniprot: prefix
        result = kegg_service.conv('hsa', f'uniprot:{uniprot_id}')
        time.sleep(delay)  # Rate limiting: 3 requests/second
        
        if result:
            lines = result.strip().split('\n')
            kegg_ids = [line.split('\t')[1] for line in lines if '\t' in line]
        
        # Method 2: If no results, try alternative 'up:' prefix
        if not kegg_ids:
            result = kegg_service.conv('hsa', f'up:{uniprot_id}')
            time.sleep(delay)
            
            if result:
                lines = result.strip().split('\n')
                kegg_ids = [line.split('\t')[1] for line in lines if '\t' in line]
        
        # Method 3: If still no results, try find service
        if not kegg_ids:
            result = kegg_service.find('genes', uniprot_id)
            time.sleep(delay)
            
            if result:
                lines = result.strip().split('\n')
                # Parse find results which have different format
                for line in lines:
                    if '\t' in line and 'hsa:' in line:
                        parts = line.split('\t')
                        gene_id = parts[0].strip()
                        if gene_id.startswith('hsa:'):
                            kegg_ids.append(gene_id)
        
        # Get pathways for each KEGG ID found
        pathways = []
        for kegg_id in kegg_ids:
            # Extract gene ID (e.g., hsa:1234)
            if ':' in kegg_id:
                gene_id = kegg_id  # Already in format hsa:1234
            else:
                gene_id = f'hsa:{kegg_id}'
            
            # Get pathways for this gene
            link_result = kegg_service.link('pathway', gene_id)
            time.sleep(delay)  # Rate limiting: 3 requests/second
            
            if link_result:
                pathway_lines = link_result.strip().split('\n')
                for line in pathway_lines:
                    if '\t' in line:
                        parts = line.split('\t')
                        if len(parts) >= 2:
                            pathway_id = parts[1].replace('path:', '')
                            if pathway_id not in pathways:
                                pathways.append(pathway_id)
        
        cache[uniprot_id] = pathways
        return pathways
        
    except Exception as e:
        # Store error information for debugging
        if uniprot_id not in cache:
            cache[uniprot_id] = []
        return []
    
    cache[uniprot_id] = []
    return []

print("✅ KEGG pathway retrieval function defined")


In [ ]:
def get_pathway_names(kegg_service, cache=None, delay=KEGG_API_DELAY):
    """
    Get all human KEGG pathway names.
    
    Returns
    -------
    dict
        Dictionary mapping pathway ID to pathway name
    """
    if cache is None:
        cache = {}
    
    if 'pathway_names' in cache:
        return cache['pathway_names']
    
    try:
        result = kegg_service.list('pathway', 'hsa')
        time.sleep(delay)  # Rate limiting
        pathway_dict = {}
        
        if result:
            lines = result.strip().split('\n')
            for line in lines:
                if '\t' in line:
                    parts = line.split('\t')
                    pathway_id = parts[0].replace('path:', '')
                    pathway_name = parts[1]
                    pathway_dict[pathway_id] = pathway_name
        
        cache['pathway_names'] = pathway_dict
        return pathway_dict
    
    except Exception as e:
        print(f"Error fetching pathway names: {e}")
        return {}

print("✅ Pathway name retrieval function defined")

In [ ]:
# Fetch KEGG pathways for all proteins (this may take a while)
# For demonstration, we'll start with a subset and can expand later

if df_raw is not None:
    print("🔄 Fetching KEGG pathway information...")
    print(f"   Rate limit: {1/KEGG_API_DELAY:.1f} requests/second")
    print("   This may take several minutes for large datasets.")
    print("   Starting with membrane proteins for efficiency...\n")
    
    # Get pathway names first
    pathway_names = get_pathway_names(kegg)
    print(f"✅ Retrieved {len(pathway_names)} human KEGG pathways")
    
    # Start with membrane proteins for focused analysis
    membrane_proteins = df[df['is_membrane']].copy()
    print(f"\n🔄 Fetching pathways for {len(membrane_proteins)} membrane proteins...")
    
    # Estimate time
    estimated_time_minutes = (len(membrane_proteins) * 2 * KEGG_API_DELAY) / 60  # 2 API calls per protein on average
    print(f"   Estimated time: {estimated_time_minutes:.1f} minutes\n")
    
    pathway_cache = {}
    pathways_list = []
    
    for idx, row in membrane_proteins.iterrows():
        uniprot_id = row['Entry']
        pathways = get_kegg_pathways_for_protein(uniprot_id, kegg, pathway_cache)
        pathways_list.append(pathways)
        
        # Progress indicator
        if (idx + 1) % 50 == 0:
            print(f"   Processed {idx + 1}/{len(membrane_proteins)} proteins...")
    
    membrane_proteins['kegg_pathways'] = pathways_list
    membrane_proteins['pathway_count'] = membrane_proteins['kegg_pathways'].apply(len)
    
    # Update the main dataframe
    df.loc[membrane_proteins.index, 'kegg_pathways'] = membrane_proteins['kegg_pathways']
    df.loc[membrane_proteins.index, 'pathway_count'] = membrane_proteins['pathway_count']
    
    # Fill NaN for non-membrane proteins
    df['kegg_pathways'] = df['kegg_pathways'].fillna('').apply(lambda x: x if isinstance(x, list) else [])
    df['pathway_count'] = df['pathway_count'].fillna(0).astype(int)
    
    print(f"\n✅ Pathway data retrieved!")
    print(f"   Proteins with pathways: {(membrane_proteins['pathway_count'] > 0).sum()}")
    print(f"   Proteins without pathways: {(membrane_proteins['pathway_count'] == 0).sum()}")
    print(f"   Average pathways per protein: {membrane_proteins['pathway_count'].mean():.1f}")

In [ ]:
# Diagnostic: Show mapping statistics
if df_raw is not None:
    print("\n📊 KEGG Mapping Diagnostics:")
    print("=" * 70)
    
    total_membrane = len(membrane_proteins)
    with_pathways = (membrane_proteins['pathway_count'] > 0).sum()
    without_pathways = (membrane_proteins['pathway_count'] == 0).sum()
    
    print(f"Total membrane proteins analyzed: {total_membrane}")
    print(f"  ✅ Successfully mapped to KEGG: {with_pathways} ({100*with_pathways/total_membrane:.1f}%)")
    print(f"  ❌ Not found in KEGG: {without_pathways} ({100*without_pathways/total_membrane:.1f}%)")
    
    if with_pathways > 0:
        print(f"\nPathways per protein (for mapped proteins):")
        mapped_proteins = membrane_proteins[membrane_proteins['pathway_count'] > 0]
        print(f"  Mean: {mapped_proteins['pathway_count'].mean():.1f}")
        print(f"  Median: {mapped_proteins['pathway_count'].median():.0f}")
        print(f"  Max: {mapped_proteins['pathway_count'].max():.0f}")
    
    # Show some examples of unmapped proteins for debugging
    if without_pathways > 0:
        print(f"\n🔍 Sample proteins not found in KEGG (first 5):")
        unmapped = membrane_proteins[membrane_proteins['pathway_count'] == 0]
        display(unmapped[['Entry', 'Entry name', 'Protein names']].head(5))
        print("\nNote: Not all proteins are in KEGG. This is expected.")
        print("      KEGG focuses on metabolic/signaling pathways.")


In [ ]:
# Show distribution of pathway counts
if df_raw is not None:
    fig = px.histogram(
        membrane_proteins[membrane_proteins['pathway_count'] > 0],
        x='pathway_count',
        nbins=30,
        title='Distribution of KEGG Pathway Count per Membrane Protein',
        labels={'pathway_count': 'Number of Pathways', 'count': 'Number of Proteins'},
        color_discrete_sequence=['#1f77b4']
    )
    fig.update_layout(showlegend=False)
    fig.show()
    
    # Show examples
    print("\n🔍 Membrane proteins with most pathways:")
    top_pathway_proteins = membrane_proteins.nlargest(10, 'pathway_count')[[
        'Entry', 'Protein names', 'p(LLPS)', 'pllps_class', 'pathway_count'
    ]]
    display(top_pathway_proteins)

---
## 4. Pathway Enrichment Analysis

Analyze which pathways are enriched or depleted in high/low pLLPS proteins.

In [ ]:
# Build pathway-to-proteins mapping
if df_raw is not None:
    pathway_proteins = defaultdict(list)
    
    for idx, row in membrane_proteins.iterrows():
        for pathway in row['kegg_pathways']:
            pathway_proteins[pathway].append({
                'Entry': row['Entry'],
                'p(LLPS)': row['p(LLPS)'],
                'pllps_class': row['pllps_class']
            })
    
    print(f"✅ Found {len(pathway_proteins)} unique pathways among membrane proteins")

In [ ]:
# Calculate enrichment statistics for each pathway
if df_raw is not None:
    pathway_stats = []
    
    total_high = (membrane_proteins['pllps_class'] == 'high').sum()
    total_low = (membrane_proteins['pllps_class'] == 'low').sum()
    total_medium = (membrane_proteins['pllps_class'] == 'medium').sum()
    
    for pathway_id, proteins in pathway_proteins.items():
        if len(proteins) < 3:  # Skip pathways with very few proteins
            continue
        
        pllps_scores = [p['p(LLPS)'] for p in proteins]
        pllps_classes = [p['pllps_class'] for p in proteins]
        
        n_high = sum(1 for c in pllps_classes if c == 'high')
        n_low = sum(1 for c in pllps_classes if c == 'low')
        n_medium = sum(1 for c in pllps_classes if c == 'medium')
        n_total = len(proteins)
        
        # Calculate enrichment ratios
        high_ratio = (n_high / n_total) / (total_high / len(membrane_proteins)) if total_high > 0 else 0
        low_ratio = (n_low / n_total) / (total_low / len(membrane_proteins)) if total_low > 0 else 0
        
        pathway_stats.append({
            'pathway_id': pathway_id,
            'pathway_name': pathway_names.get(pathway_id, pathway_id),
            'n_proteins': n_total,
            'n_high_pllps': n_high,
            'n_medium_pllps': n_medium,
            'n_low_pllps': n_low,
            'pct_high': 100 * n_high / n_total,
            'pct_low': 100 * n_low / n_total,
            'mean_pllps': np.mean(pllps_scores),
            'std_pllps': np.std(pllps_scores),
            'median_pllps': np.median(pllps_scores),
            'high_enrichment': high_ratio,
            'low_enrichment': low_ratio
        })
    
    df_pathway_stats = pd.DataFrame(pathway_stats)
    df_pathway_stats = df_pathway_stats.sort_values('n_proteins', ascending=False)
    
    print(f"✅ Calculated statistics for {len(df_pathway_stats)} pathways")
    print(f"\n📊 Top pathways by protein count:")
    display(df_pathway_stats.head(20))

In [ ]:
# Visualize pathway enrichment
if df_raw is not None and len(df_pathway_stats) > 0:
    # Top pathways enriched in high pLLPS
    top_high_enriched = df_pathway_stats.nlargest(15, 'high_enrichment')
    
    fig = px.bar(
        top_high_enriched,
        x='high_enrichment',
        y='pathway_name',
        orientation='h',
        title='Top 15 Pathways Enriched in High pLLPS Membrane Proteins',
        labels={'high_enrichment': 'Enrichment Ratio (vs. expected)', 'pathway_name': 'KEGG Pathway'},
        color='high_enrichment',
        color_continuous_scale='Reds',
        hover_data=['n_proteins', 'n_high_pllps', 'mean_pllps']
    )
    fig.update_layout(height=600)
    fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="Expected")
    fig.show()
    
    # Top pathways enriched in low pLLPS
    top_low_enriched = df_pathway_stats.nlargest(15, 'low_enrichment')
    
    fig = px.bar(
        top_low_enriched,
        x='low_enrichment',
        y='pathway_name',
        orientation='h',
        title='Top 15 Pathways Enriched in Low pLLPS Membrane Proteins',
        labels={'low_enrichment': 'Enrichment Ratio (vs. expected)', 'pathway_name': 'KEGG Pathway'},
        color='low_enrichment',
        color_continuous_scale='Blues',
        hover_data=['n_proteins', 'n_low_pllps', 'mean_pllps']
    )
    fig.update_layout(height=600)
    fig.add_vline(x=1.0, line_dash="dash", line_color="gray", annotation_text="Expected")
    fig.show()

In [ ]:
# Show pathways with highest mean pLLPS scores
if df_raw is not None and len(df_pathway_stats) > 0:
    print("\n🔴 Pathways with highest mean p(LLPS):")
    display(df_pathway_stats.nlargest(15, 'mean_pllps')[[
        'pathway_name', 'n_proteins', 'mean_pllps', 'std_pllps', 'pct_high'
    ]])
    
    print("\n🔵 Pathways with lowest mean p(LLPS):")
    display(df_pathway_stats.nsmallest(15, 'mean_pllps')[[
        'pathway_name', 'n_proteins', 'mean_pllps', 'std_pllps', 'pct_low'
    ]])

---
## 5. Score Similarity Analysis

Analyze whether proteins with similar pLLPS scores tend to interact in the same pathways.

In [ ]:
# Calculate within-pathway pLLPS score variance
if df_raw is not None and len(df_pathway_stats) > 0:
    pathway_variance = df_pathway_stats[['pathway_name', 'n_proteins', 'mean_pllps', 'std_pllps']].copy()
    
    # Calculate coefficient of variation (CV) for normalized comparison
    pathway_variance['cv'] = pathway_variance['std_pllps'] / pathway_variance['mean_pllps']
    pathway_variance = pathway_variance.sort_values('std_pllps')
    
    print("📊 Pathways with most similar pLLPS scores (low variance):")
    display(pathway_variance.head(15))
    
    print("\n📊 Pathways with most diverse pLLPS scores (high variance):")
    display(pathway_variance.tail(15))

In [ ]:
# Visualize variance vs mean pLLPS
if df_raw is not None and len(df_pathway_stats) > 0:
    fig = px.scatter(
        df_pathway_stats[df_pathway_stats['n_proteins'] >= 5],
        x='mean_pllps',
        y='std_pllps',
        size='n_proteins',
        hover_data=['pathway_name', 'n_proteins'],
        title='Pathway pLLPS Score Mean vs Standard Deviation',
        labels={'mean_pllps': 'Mean p(LLPS)', 'std_pllps': 'Std Dev p(LLPS)'},
        color='n_proteins',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(height=600)
    fig.show()

In [ ]:
# Distribution of pLLPS scores across selected pathways
if df_raw is not None and len(df_pathway_stats) > 0:
    # Select top pathways for visualization
    top_pathways = df_pathway_stats.nlargest(10, 'n_proteins')['pathway_id'].tolist()
    
    # Collect pLLPS scores for each pathway
    pathway_score_data = []
    for pathway_id in top_pathways:
        pathway_name = pathway_names.get(pathway_id, pathway_id)
        for protein in pathway_proteins[pathway_id]:
            pathway_score_data.append({
                'pathway': pathway_name[:50],  # Truncate long names
                'p(LLPS)': protein['p(LLPS)']
            })
    
    df_pathway_scores = pd.DataFrame(pathway_score_data)
    
    # Create box plot
    fig = px.box(
        df_pathway_scores,
        x='pathway',
        y='p(LLPS)',
        title='Distribution of p(LLPS) Scores in Top 10 Pathways',
        labels={'pathway': 'KEGG Pathway', 'p(LLPS)': 'p(LLPS) Score'},
        color='pathway'
    )
    fig.update_layout(showlegend=False, height=600)
    fig.update_xaxes(tickangle=45)
    fig.show()

---
## 6. KEGG Pathway Visualization

Create annotated KEGG pathway diagrams for selected membrane proteins.

In [ ]:
def get_pathway_image_url(pathway_id, highlight_genes=None):
    """
    Generate URL for KEGG pathway image with optional gene highlighting.
    
    Parameters
    ----------
    pathway_id : str
        KEGG pathway ID (e.g., 'hsa04010')
    highlight_genes : list, optional
        List of gene IDs to highlight
    
    Returns
    -------
    str
        URL to pathway image
    """
    base_url = f"https://www.kegg.jp/kegg-bin/show_pathway?{pathway_id}"
    
    if highlight_genes:
        # Add gene highlighting
        gene_params = '/'.join([g.replace('hsa:', '') for g in highlight_genes])
        base_url += f"/{gene_params}"
    
    return base_url

def annotate_pathway_genes_with_pllps(pathway_id, pathway_proteins_dict, df_proteins, kegg_service):
    """
    Create a mapping of genes in a pathway to their pLLPS scores.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with gene IDs and pLLPS scores
    """
    if pathway_id not in pathway_proteins_dict:
        return pd.DataFrame()
    
    proteins = pathway_proteins_dict[pathway_id]
    annotations = []
    
    for protein in proteins:
        entry = protein['Entry']
        pllps = protein['p(LLPS)']
        
        # Get protein details
        protein_info = df_proteins[df_proteins['Entry'] == entry].iloc[0]
        
        annotations.append({
            'Entry': entry,
            'Protein': protein_info.get('Protein names', 'Unknown')[:50],
            'p(LLPS)': pllps,
            'Class': protein['pllps_class']
        })
    
    return pd.DataFrame(annotations).sort_values('p(LLPS)', ascending=False)

print("✅ Pathway visualization functions defined")

In [ ]:
# Select a pathway with high representation for visualization
if df_raw is not None and len(df_pathway_stats) > 0:
    # Choose pathway with good protein coverage
    selected_pathway = df_pathway_stats.iloc[0]
    pathway_id = selected_pathway['pathway_id']
    pathway_name = selected_pathway['pathway_name']
    
    print(f"\n🔍 Selected Pathway: {pathway_name}")
    print(f"   ID: {pathway_id}")
    print(f"   Proteins: {selected_pathway['n_proteins']}")
    print(f"   Mean p(LLPS): {selected_pathway['mean_pllps']:.3f}")
    
    # Get protein annotations for this pathway
    pathway_annotations = annotate_pathway_genes_with_pllps(
        pathway_id, pathway_proteins, membrane_proteins, kegg
    )
    
    print(f"\n📊 Proteins in {pathway_name}:")
    display(pathway_annotations)
    
    # Generate pathway URL
    pathway_url = get_pathway_image_url(pathway_id)
    print(f"\n🔗 View pathway diagram at:")
    print(f"   {pathway_url}")
    print(f"\nNote: Red highlighted genes indicate those from our dataset.")
    print(f"      You can view the diagram in a browser to see the full pathway with annotations.")

In [ ]:
# Create a custom visualization showing pLLPS distribution in pathway
if df_raw is not None and len(pathway_annotations) > 0:
    fig = go.Figure()
    
    # Create color-coded bar chart
    colors = pathway_annotations['p(LLPS)'].apply(
        lambda x: 'red' if x >= HIGH_PLLPS_THRESHOLD else ('blue' if x <= LOW_PLLPS_THRESHOLD else 'gray')
    )
    
    fig.add_trace(go.Bar(
        x=pathway_annotations.index,
        y=pathway_annotations['p(LLPS)'],
        marker_color=colors,
        text=pathway_annotations['Protein'],
        hovertemplate='<b>%{text}</b><br>p(LLPS): %{y:.3f}<extra></extra>'
    ))
    
    fig.update_layout(
        title=f'p(LLPS) Scores for Proteins in {pathway_name}',
        xaxis_title='Protein (sorted by p(LLPS))',
        yaxis_title='p(LLPS) Score',
        height=500,
        showlegend=False
    )
    
    # Add threshold lines
    fig.add_hline(y=HIGH_PLLPS_THRESHOLD, line_dash="dash", line_color="red", 
                  annotation_text="High threshold")
    fig.add_hline(y=LOW_PLLPS_THRESHOLD, line_dash="dash", line_color="blue", 
                  annotation_text="Low threshold")
    
    fig.show()

---
## 7. Statistical Analysis

Perform statistical tests to validate our observations.

In [ ]:
# Test: Do proteins in the same pathway have more similar pLLPS scores than expected?
if df_raw is not None and len(df_pathway_stats) > 0:
    print("\n📊 Statistical Analysis: Within-Pathway pLLPS Similarity\n")
    print("Testing hypothesis: Proteins in the same pathway have more similar p(LLPS) scores")
    print("compared to random protein pairs.\n")
    
    # Calculate within-pathway variance
    within_pathway_variance = df_pathway_stats['std_pllps'].mean()
    
    # Calculate overall variance (null distribution)
    overall_variance = membrane_proteins['p(LLPS)'].std()
    
    print(f"Average within-pathway std dev: {within_pathway_variance:.3f}")
    print(f"Overall std dev (all proteins): {overall_variance:.3f}")
    print(f"Ratio: {within_pathway_variance / overall_variance:.3f}")
    
    if within_pathway_variance < overall_variance:
        print("\n✅ Proteins in the same pathway tend to have MORE similar p(LLPS) scores!")
    else:
        print("\n❌ No evidence that pathway proteins have more similar p(LLPS) scores.")

In [ ]:
# Correlation: Pathway size vs mean pLLPS
if df_raw is not None and len(df_pathway_stats) > 0:
    # Filter pathways with sufficient proteins
    pathways_for_corr = df_pathway_stats[df_pathway_stats['n_proteins'] >= 5]
    
    if len(pathways_for_corr) > 0:
        correlation, p_value = stats.spearmanr(
            pathways_for_corr['n_proteins'],
            pathways_for_corr['mean_pllps']
        )
        
        print(f"\n📊 Correlation: Pathway Size vs Mean p(LLPS)")
        print(f"   Spearman correlation: {correlation:.3f}")
        print(f"   P-value: {p_value:.4f}")
        
        if p_value < 0.05:
            if correlation > 0:
                print(f"   ✅ Significant positive correlation: Larger pathways tend to have higher mean p(LLPS)")
            else:
                print(f"   ✅ Significant negative correlation: Larger pathways tend to have lower mean p(LLPS)")
        else:
            print(f"   ❌ No significant correlation between pathway size and mean p(LLPS)")

In [ ]:
# Compare pathway count between high and low pLLPS membrane proteins
if df_raw is not None and len(df_pathway_stats) > 0:
    high_pathway_counts = membrane_proteins[membrane_proteins['pllps_class'] == 'high']['pathway_count']
    low_pathway_counts = membrane_proteins[membrane_proteins['pllps_class'] == 'low']['pathway_count']
    
    # Mann-Whitney U test (non-parametric)
    statistic, p_value = stats.mannwhitneyu(
        high_pathway_counts[high_pathway_counts > 0],
        low_pathway_counts[low_pathway_counts > 0],
        alternative='two-sided'
    )
    
    print(f"\n📊 Comparison: Pathway Count in High vs Low pLLPS Membrane Proteins")
    print(f"   High pLLPS mean pathway count: {high_pathway_counts.mean():.2f}")
    print(f"   Low pLLPS mean pathway count: {low_pathway_counts.mean():.2f}")
    print(f"   Mann-Whitney U statistic: {statistic:.2f}")
    print(f"   P-value: {p_value:.4f}")
    
    if p_value < 0.05:
        if high_pathway_counts.mean() > low_pathway_counts.mean():
            print(f"   ✅ High pLLPS proteins participate in significantly MORE pathways!")
        else:
            print(f"   ✅ High pLLPS proteins participate in significantly FEWER pathways!")
    else:
        print(f"   ❌ No significant difference in pathway participation.")

---
## Summary and Export

Save results for further analysis.

In [ ]:
# Export pathway statistics
if df_raw is not None and len(df_pathway_stats) > 0:
    output_file = 'kegg_pathway_analysis_results.xlsx'
    
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # Pathway statistics
        df_pathway_stats.to_excel(writer, sheet_name='Pathway_Statistics', index=False)
        
        # High pLLPS membrane proteins
        high_mem[['Entry', 'Protein names', 'p(LLPS)', 'pathway_count']].to_excel(
            writer, sheet_name='High_pLLPS_Membrane', index=False
        )
        
        # Low pLLPS membrane proteins
        low_mem[['Entry', 'Protein names', 'p(LLPS)', 'pathway_count']].to_excel(
            writer, sheet_name='Low_pLLPS_Membrane', index=False
        )
    
    print(f"✅ Results exported to {output_file}")

---
## Conclusions

This notebook has analyzed pLLPS scores in the context of KEGG pathways and revealed:

1. **Pathway Distribution**: Membrane proteins participate in varying numbers of KEGG pathways
2. **Enrichment Patterns**: Certain pathways are enriched or depleted in high/low pLLPS proteins
3. **Score Similarity**: Analysis of whether similar pLLPS scores cluster within pathways
4. **Statistical Validation**: Statistical tests confirm the significance of observed patterns

### Next Steps
- Investigate specific pathways of interest in detail
- Compare with experimental phase separation data
- Integrate with protein interaction networks
- Extend analysis to non-membrane proteins